# Imports

In [13]:
import kagglehub
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load in data

In [ ]:
path = kagglehub.dataset_download("alistairking/weather-long-term-time-series-forecasting")

['cleaned_weather.csv']
Path to dataset files: C:\Users\Dell\.cache\kagglehub\datasets\alistairking\weather-long-term-time-series-forecasting\versions\1


In [8]:
df = pd.read_csv(f"{path}\\cleaned_weather.csv")
df.head()

,date,p,T,Tpot,Tdew,rh,VPmax,VPact,VPdef,sh,...,rho,wv,max. wv,wd,rain,raining,SWDR,PAR,max. PAR,Tlog
0,2020-01-01 00:10:00,1008.89,0.71,273.18,-1.33,86.1,6.43,5.54,0.89,3.42,...,1280.62,1.02,1.60,224.3,0.0,0.0,0.0,0.0,0.0,11.45
1,2020-01-01 00:20:00,1008.76,0.75,273.22,-1.44,85.2,6.45,5.49,0.95,3.39,...,1280.33,0.43,0.84,206.8,0.0,0.0,0.0,0.0,0.0,11.51
2,2020-01-01 00:30:00,1008.66,0.73,273.21,-1.48,85.1,6.44,5.48,0.96,3.39,...,1280.29,0.61,1.48,197.1,0.0,0.0,0.0,0.0,0.0,11.60
3,2020-01-01 00:40:00,1008.64,0.37,272.86,-1.64,86.3,6.27,5.41,0.86,3.35,...,1281.97,1.11,1.48,206.4,0.0,0.0,0.0,0.0,0.0,11.70
4,2020-01-01 00:50:00,1008.61,0.33,272.82,-1.50,87.4,6.26,5.47,0.79,3.38,...,1282.08,0.49,1.40,209.6,0.0,0.0,0.0,0.0,0.0,11.81


# Data cleaning and engineering

## Check for missing values

In [9]:
df.isnull().sum()

date        0
p           0
T           0
Tpot        0
Tdew        0
rh          0
VPmax       0
VPact       0
VPdef       0
sh          0
H2OC        0
rho         0
wv          0
max. wv     0
wd          0
rain        0
raining     0
SWDR        0
PAR         0
max. PAR    0
Tlog        0
dtype: int64

No missing values

## Check types

In [10]:
df.dtypes

date         object
p           float64
T           float64
Tpot        float64
Tdew        float64
rh          float64
VPmax       float64
VPact       float64
VPdef       float64
sh          float64
H2OC        float64
rho         float64
wv          float64
max. wv     float64
wd          float64
rain        float64
raining     float64
SWDR        float64
PAR         float64
max. PAR    float64
Tlog        float64
dtype: object

`date` feature is of type object, we convert it to datetime.

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df.dtypes

date        datetime64[ns]
p                  float64
T                  float64
Tpot               float64
Tdew               float64
rh                 float64
VPmax              float64
VPact              float64
VPdef              float64
sh                 float64
H2OC               float64
rho                float64
wv                 float64
max. wv            float64
wd                 float64
rain               float64
raining            float64
SWDR               float64
PAR                float64
max. PAR           float64
Tlog               float64
dtype: object

## Trigonometric features based on time

In [14]:
df["hour_sin"] = np.sin(2 * np.pi * df["date"].dt.hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["date"].dt.hour / 24)
df["month_sin"] = np.sin(2 * np.pi * df["date"].dt.month / 12)
df["month_cos"] = np.cos(2 * np.pi * df["date"].dt.month / 12)

df = df.drop(columns=["date"])
df.head()

,p,T,Tpot,Tdew,rh,VPmax,VPact,VPdef,sh,H2OC,...,rain,raining,SWDR,PAR,max. PAR,Tlog,hour_sin,hour_cos,month_sin,month_cos
0,1008.89,0.71,273.18,-1.33,86.1,6.43,5.54,0.89,3.42,5.49,...,0.0,0.0,0.0,0.0,0.0,11.45,0.0,1.0,0.5,0.866025
1,1008.76,0.75,273.22,-1.44,85.2,6.45,5.49,0.95,3.39,5.45,...,0.0,0.0,0.0,0.0,0.0,11.51,0.0,1.0,0.5,0.866025
2,1008.66,0.73,273.21,-1.48,85.1,6.44,5.48,0.96,3.39,5.43,...,0.0,0.0,0.0,0.0,0.0,11.60,0.0,1.0,0.5,0.866025
3,1008.64,0.37,272.86,-1.64,86.3,6.27,5.41,0.86,3.35,5.37,...,0.0,0.0,0.0,0.0,0.0,11.70,0.0,1.0,0.5,0.866025
4,1008.61,0.33,272.82,-1.50,87.4,6.26,5.47,0.79,3.38,5.42,...,0.0,0.0,0.0,0.0,0.0,11.81,0.0,1.0,0.5,0.866025


# Train Val Test split

## Split

In [16]:
n = len(df)
train_df = df[0:int(n*0.7)]
val_df = df[int(n*0.7):int(n*0.9)]
test_df = df[int(n*0.9):]

## Scaling

In [17]:
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_df)
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

## Creating windows

In [18]:
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size, 0])
    return np.array(X), np.array(y)

window = 72
X_train, y_train = create_sequences(train_scaled, window)
X_val, y_val = create_sequences(val_scaled, window)
X_test, y_test = create_sequences(test_scaled, window)